# Where do targets come from?

**Optional code walkthrough for AI 101 (Module 4).** Not graded. Quiz scores do not depend on it.

**Objective.** See, in small plain-Python examples, where the learning signal comes from in three setups: given targets (supervised), targets built from the data (self-supervised) and reward from interaction (reinforcement learning). Nothing is trained here.

**Supports.** Lesson 4.1 (section “Supervised learning uses input-target pairs”), Lesson 4.2 (section “Masked-signal reconstruction”) and Lesson 4.3 (sections “Simulated thermal controller” and “Rewards and safety limits”).

Run the cells from top to bottom. All data are synthetic and invented for teaching.


## About the data

You use a few invented numbers in three small examples. They are synthetic and have no link to a real machine.

Example 1 shows input-target pairs. Example 2 hides part of a signal and keeps the hidden values as targets. Example 3 runs a very small simulated heater for a few steps, with a fixed rule instead of learning.

```
Example 1 (supplied targets): flow (m³/s), pressure rise (Pa) -> power (W)
Example 2 (built targets):    signal in °C = 20, 21, 22, 23, 24, 25
Example 3 (interaction):      start 296 K, target 300 K, safety limit 305 K
```


## Step 1. Supervised: the targets are supplied

In supervised learning each example comes with a known target (Lesson 4.1). Here each target is a number, a measured power in W, so this would be a regression problem. If the targets were categories such as intact or cracked, it would be classification.

In [ ]:
# Step 1: input-target pairs (synthetic)
# inputs: (flow in m³/s, pressure rise in Pa); target: measured power in W
pairs = [
    ((0.020, 100000), 2800),
    ((0.015,  80000), 1650),
    ((0.025, 120000), 3950),
]

for inputs, target in pairs:
    print("inputs", inputs, "-> given target", target, "W")

**Expected output**

```
inputs (0.02, 100000) -> given target 2800 W
inputs (0.015, 80000) -> given target 1650 W
inputs (0.025, 120000) -> given target 3950 W
```

A person or a measurement supplied each target. Training would adjust a model until its outputs come close to these targets.

## Step 2. Self-supervised: build the target from the data

In masked-signal reconstruction (Lesson 4.2) you hide a section of a signal. The hidden values are the target, and the original signal supplies them. Nobody labels anything by hand. A real model would estimate the hidden values. To keep the code simple, a fixed rule stands in for the model: it repeats the last visible value.

In [ ]:
# Step 2: build a constructed input and a constructed target from one signal
signal = [20.0, 21.0, 22.0, 23.0, 24.0, 25.0]    # observed signal in °C (synthetic)
hidden = [2, 3]                                   # positions we hide

constructed_input = []
for i in range(len(signal)):
    if i in hidden:
        constructed_input.append(None)            # None means "hidden"
    else:
        constructed_input.append(signal[i])

constructed_target = []
for i in hidden:
    constructed_target.append(signal[i])          # original values of the hidden part
print("constructed input: ", constructed_input)
print("constructed target:", constructed_target)

# a fixed stand-in for a model: repeat the last visible value
estimate = [constructed_input[1], constructed_input[1]]
print("estimate:          ", estimate)
print("target - estimate: ", [constructed_target[0] - estimate[0], constructed_target[1] - estimate[1]])

**Expected output**

```
constructed input:  [20.0, 21.0, None, None, 24.0, 25.0]
constructed target: [22.0, 23.0]
estimate:           [21.0, 21.0]
target - estimate:  [1.0, 2.0]
```

The constructed input has a gap. The constructed target keeps the original hidden values, 22.0 and 23.0 °C. The last line shows the disagreement between the target and the estimate. Training aims to make this disagreement smaller. Inputs, targets and an objective all exist even though nobody labelled the data (Lesson 4.2).

## Step 3. Reinforcement learning: the roles in a loop

In reinforcement learning an agent sees a state, picks an action and receives a reward from an environment (Lesson 4.3). Here the state is the temperature, the action is the heater command (1 = on, 0 = off), and the environment is a made-up thermal rule. The policy is a fixed rule you write: heat when below the target. Nothing learns. The reward and the safety limit are kept separate. All numbers, including the reward weights, are invented for teaching.

In [ ]:
# Step 3: a tiny simulated thermal loop (synthetic, bounded simulation)
temp = 296.0              # state: temperature in K
target = 300.0            # target temperature in K
safety_limit = 305.0      # safety constraint in K (kept separate from the reward)
heat_step = 2.0           # K gained per step when the heater is on
loss_step = 0.5           # K lost per step

for step in range(6):
    action = 1 if temp < target else 0          # policy: heater on below the target
    temp = temp + heat_step * action - loss_step   # environment responds
    reward = -abs(temp - target) - 0.2 * action    # tracking error plus an energy penalty
    safe = temp <= safety_limit
    print(f"step {step}: action {action}, temperature {temp:.1f} K, reward {reward:.1f}, within limit: {safe}")

**Expected output**

```
step 0: action 1, temperature 297.5 K, reward -2.7, within limit: True
step 1: action 1, temperature 299.0 K, reward -1.2, within limit: True
step 2: action 1, temperature 300.5 K, reward -0.7, within limit: True
step 3: action 0, temperature 300.0 K, reward -0.0, within limit: True
step 4: action 0, temperature 299.5 K, reward -0.5, within limit: True
step 5: action 1, temperature 301.0 K, reward -1.2, within limit: True
```

Each row is one pass through state, action, environment and reward. The reward is higher (closer to 0) when the temperature is close to the target and the heater uses less. The “within limit” column is a separate safety check. A good reward does not replace it (Lesson 4.3). The “-0.0” in step 3 is only how Python prints a negative zero.

## Try it

Change a value and observe. In the Step 3 code, change `heat_step = 2.0` to `heat_step = 6.0`. Then run Step 3 again.

**Expected result**

```
step 0: action 1, temperature 301.5 K, reward -1.7, within limit: True
step 1: action 0, temperature 301.0 K, reward -1.0, within limit: True
step 2: action 0, temperature 300.5 K, reward -0.5, within limit: True
step 3: action 0, temperature 300.0 K, reward -0.0, within limit: True
step 4: action 0, temperature 299.5 K, reward -0.5, within limit: True
step 5: action 1, temperature 305.0 K, reward -5.2, within limit: True
```

With a stronger heater the temperature jumps past the target. In step 5 it reaches exactly 305.0 K, the safety limit, and the reward falls to −5.2. Try `heat_step = 8.0` as well: the temperature then goes above the limit and “within limit” becomes False. The reward and the safety limit measure different things. A high reward never proves safe behaviour.

## Limits

- Nothing in this walkthrough learns. The “model” in Step 2 and the policy in Step 3 are fixed rules that stand in for a learned model.
- The temperature rule, the reward weights and the limit in Step 3 are invented. They are not a design for a real heater controller.
- The course does not authorise experiments on real machinery. Use bounded simulations only (Lesson 4.3).
- Clustering and anomaly detection (Lesson 4.1) are not shown here. Clustering appears in Module 9.

